In [ ]:
!pip install ipywidgets --upgrade

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset,DataLoader
from transformers import BertTokenizer,BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score,f1_score,classification_report
import warnings
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
df_train=pd.read_csv('train_clean.csv')
df_test=pd.read_csv('test_clean.csv')
df_dev=pd.read_csv('dev_clean.csv')
print(f"Train: {len(df_train)} | Test: {len(df_test)} | Dev: {len(df_dev)}")
print(df_train.head(3))

In [ ]:
df_train_13 = df_train.copy()
df_test_13  = df_test.copy()
df_dev_13  = df_dev.copy()

print(f'Train: {len(df_train_13)} | Test: {len(df_test_13)} | Dev: {len(df_dev_13)}')
print('\nLabel distribution:')
print(df_train_13['label'].value_counts())


In [ ]:
le = LabelEncoder()
df_train_13['label_id'] = le.fit_transform(df_train_13['label'])
df_test_13['label_id']  = le.transform(df_test_13['label'])
df_dev_13['label_id']   = le.transform(df_dev_13['label'])

label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print('Label mapping:', label_mapping)
num_classes = len(le.classes_)
print(f'Number of classes: {num_classes}')

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

class FallacyDataset(Dataset):
    def __init__(self,texts,labels,tokenizer,max_len=128):
        self.texts=texts
        self.labels=labels
        self.tokenizer=tokenizer
        self.max_len=max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,indx):
        encoding=self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':encoding['input_ids'].squeeze(),
            'attention_mask':encoding['attention_mask'].squeeze(),
            'label':torch.tensor(self.labels[idx],dtype=torch.long)
        }
#datasets
train_dataset = FallacyDataset(df_train_13['text'].tolist(), df_train_13['label_id'].tolist(), tokenizer)
dev_dataset   = FallacyDataset(df_dev_13['text'].tolist(),   df_dev_13['label_id'].tolist(),   tokenizer)
test_dataset  = FallacyDataset(df_test_13['text'].tolist(),  df_test_13['label_id'].tolist(),  tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"✅ Datasets ready!")
print(f"Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)}")       

In [ ]:
# ── Cell 6: Load BERT Model ────────────────────────────────────
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=num_classes)
model = model.to(device)
print(f"✅ BERT model loaded with {num_classes} output classes")

In [ ]:
# ── Weighted Loss Cell ─────────────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute class weights from training data
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df_train_13['label_id']),
    y=df_train_13['label_id'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:", dict(zip(le.classes_, class_weights.round(3))))

In [ ]:
EPOCHS       = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)  # 10% warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Total training steps : {total_steps}')
print(f'Warmup steps         : {warmup_steps}')
print(f'Epochs               : {EPOCHS}')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        logits  = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = outputs.loss
            logits  = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1, all_preds, all_labels

print('✅ Training functions ready!')

In [ ]:
best_val_f1 = 0
history     = []

print('Starting training...\n')
print(f'{"Epoch":<8} {"Train Loss":<14} {"Train Acc":<13} {"Val Loss":<12} {"Val Acc":<11} {"Val F1"}')
print('-' * 70)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, dev_loader, device)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss,     'val_acc': val_acc, 'val_f1': val_f1
    })

    print(f'{epoch:<8} {train_loss:<14.4f} {train_acc:<13.4f} {val_loss:<12.4f} {val_acc:<11.4f} {val_f1:.4f}')

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  💾 Best model saved! (Val F1: {val_f1:.4f})')

print('\n✅ Training complete!')
print(f'Best Validation F1: {best_val_f1:.4f}')

In [ ]:
model.load_state_dict(torch.load('best_model.pt'))
model.to(device)

test_loss, test_acc, test_f1, preds, labels = evaluate(model, test_loader, device)

print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test F1 Score : {test_f1:.4f}')
print('\n=== Classification Report ===')
print(classification_report(labels, preds, target_names=le.classes_))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Confusion Matrix — 5 Class BERT Baseline', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('✅ Confusion matrix saved!')

In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history_df['epoch'], history_df['val_loss'],   label='Val Loss',   marker='o')
axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history_df['epoch'], history_df['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history_df['epoch'], history_df['val_acc'],   label='Val Acc',   marker='o')
axes[1].set_title('Accuracy per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()
print('✅ Training history saved!')

In [ ]:
def predict_fallacy(text, model, tokenizer, le, device):
    model.eval()
    encoding = tokenizer(
        text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_id].item()

    return le.classes_[pred_id], round(confidence * 100, 2)

# Test with sample texts
test_texts = [
    "Everyone believes this, so it must be right.",
    "You can't trust him, he's not even qualified.",
    "Either you're with us or against us.",
    "She's always late, typical of her kind.",
    "I feel so strongly about this it must be true.",
    "This policy worked in the 80s so it will work now.",
    "The scientist said so, therefore it must be true.",
    "If we allow this, next thing you know everything will collapse.",
    "You say you care about health but you eat junk food.",
    "This word means one thing here and another thing there.",
    "That's not really what I said, you're twisting my words.",
    "We've always done it this way so it must be correct.",
    "The data clearly shows a pattern here."
]

print('=== Inference Test ===\n')
for text in test_texts:
    label, confidence = predict_fallacy(text, model, tokenizer, le, device)
    print(f'Text      : {text}')
    print(f'Prediction: {label} ({confidence}% confidence)')
    print()